## 1. Before You Begin

MAI-Image-2.5-Pro extends the base model with stronger capabilities for complex scenes:

- **High-fidelity portraits** — accurate facial structure, consistent skin tones, natural lighting
- **Accurate text rendering** — readable labels, signage, and packaging text inside generated images
- **Visual reasoning** — coherent spatial relationships, material accuracy, character consistency

**Pricing:** $5 / 1M text tokens in · $106 / 1M image tokens out —
[source](https://techcommunity.microsoft.com/blog/azure-ai-foundry-blog/introducing-mai-image-2-5-pro-and-mai-voice-2-flash-in-microsoft-foundry/4539446)

**Prerequisites**

1. A Microsoft Foundry project with MAI-Image-2.5-Pro deployed (available from 2026-07-23).
   See [models/quickstart/](../../quickstart/README.md) for first-time setup.
2. The three environment variables listed in section 2.
3. `requests` installed: `pip install requests`

**API constraints** ([source: Learn docs](https://learn.microsoft.com/en-us/azure/foundry/foundry-models/how-to/use-foundry-models-mai-image?tabs=python))

| Rule | Value |
|---|---|
| Minimum dimension (each) | 768 px |
| Maximum total pixels | 1,048,576 |
| Output format | PNG only |
| Auth header | `api-key` |


## 2. Set up your environment

The cell below verifies the three required variables and derives `BASE_ENDPOINT`.
`MICROSOFT_FOUNDRY_ENDPOINT` has the form:
```
https://<resource>.services.ai.azure.com/api/projects/<project>
```
Stripping the path suffix gives the base URL used by all MAI image endpoints.


In [ ]:
%pip install requests python-dotenv --quiet

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import os, base64
import requests
from pathlib import Path
from urllib.parse import urlparse
from IPython.display import Image as IPyImage, display

REQUIRED = {
    "MICROSOFT_FOUNDRY_ENDPOINT":           "Foundry project endpoint",
    "MICROSOFT_FOUNDRY_API_KEY":             "AIServices API key",
    "AZURE_MAI_IMAGE_25_PRO_DEPLOYMENT":     "MAI-Image-2.5-Pro deployment name",
}
missing = [k for k in REQUIRED if not os.environ.get(k)]
if missing:
    raise EnvironmentError(f"Set these env vars before continuing: {missing}")

parsed        = urlparse(os.environ["MICROSOFT_FOUNDRY_ENDPOINT"])
BASE_ENDPOINT = f"{parsed.scheme}://{parsed.netloc}"
API_KEY       = os.environ["MICROSOFT_FOUNDRY_API_KEY"]
DEPLOYMENT    = os.environ["AZURE_MAI_IMAGE_25_PRO_DEPLOYMENT"]

print(f"Base endpoint : {BASE_ENDPOINT}")
print(f"Deployment    : {DEPLOYMENT}")
print("Environment check passed.")


## 3. Configure the client

The same two helper functions from the base capsule apply here.
Pro shares identical API endpoints — the difference is in what you can ask of the model.


In [ ]:
GEN_URL  = f"{BASE_ENDPOINT}/mai/v1/images/generations"
EDIT_URL = f"{BASE_ENDPOINT}/mai/v1/images/edits"

OUT_DIR = Path("output")
OUT_DIR.mkdir(exist_ok=True)


def generate_image(prompt: str, width: int = 1024, height: int = 1024) -> str:
    """POST to /mai/v1/images/generations; return base64 PNG string."""
    assert width >= 768 and height >= 768, "Each dimension must be ≥ 768 px"
    assert width * height <= 1_048_576,    "width × height must be ≤ 1,048,576"
    resp = requests.post(
        GEN_URL,
        headers={"Content-Type": "application/json", "api-key": API_KEY},
        json={"model": DEPLOYMENT, "prompt": prompt, "width": width, "height": height},
        timeout=120,
    )
    resp.raise_for_status()
    return resp.json()["data"][0]["b64_json"]


def edit_image(image_path: str, prompt: str, size: str = "1024x1024") -> str:
    """POST to /mai/v1/images/edits (multipart); return base64 PNG string."""
    w, h = (int(d) for d in size.split("x"))
    assert w >= 768 and h >= 768, "Each dimension must be ≥ 768 px"
    assert w * h <= 1_048_576,    "width × height must be ≤ 1,048,576"
    with open(image_path, "rb") as f:
        resp = requests.post(
            EDIT_URL,
            headers={"api-key": API_KEY},
            data={"model": DEPLOYMENT, "prompt": prompt, "size": size},
            files=[("image", (Path(image_path).name, f, "image/png"))],
            timeout=120,
        )
    resp.raise_for_status()
    return resp.json()["data"][0]["b64_json"]


def show(b64: str, save_as: str | None = None) -> None:
    """Decode base64 PNG, optionally save, and display inline."""
    raw = base64.b64decode(b64)
    if save_as:
        out = OUT_DIR / save_as
        out.write_bytes(raw)
        print(f"Saved → {out}")
    display(IPyImage(data=raw))


## 4. Generate a high-fidelity portrait

Pro produces accurate facial structure, consistent skin tone, and coherent lighting.
Effective portrait prompts specify:
- **Lighting type** (natural window light, studio soft-box, golden-hour)
- **Angle and framing** (close-up headshot, three-quarter view, environmental portrait)
- **Mood or expression** (neutral, contemplative, candid)

The 768 × 1024 portrait ratio suits head-and-shoulders framing.


In [ ]:
b64_portrait = generate_image(
    prompt=(
        "Environmental portrait of a software engineer in their thirties, "
        "natural window light from the left, three-quarter view, "
        "blurred open-plan office background, candid expression, "
        "high detail, photorealistic"
    ),
    width=768,
    height=1024,
)
show(b64_portrait, save_as="01-portrait.png")


## 5. Render accurate text inside an image

One of the clearest differentiators for Pro is the ability to place readable text
in generated images — useful for packaging mockups, signage, and UI wireframes.

Tips for accurate text rendering:
- Quote the exact text in the prompt using double quotes: `"Hello World"`
- Specify the font style, size, and placement context
- Keep strings short (2–5 words render more reliably than sentences)


In [ ]:
b64_signage = generate_image(
    prompt=(
        'Storefront window sign reading "OPEN DAILY" in bold sans-serif lettering, '
        "painted white on dark navy glass, warm interior light visible behind, "
        "shallow depth of field, photorealistic"
    ),
    width=1024,
    height=768,
)
show(b64_signage, save_as="02-signage.png")


## 6. Describe a complex scene with spatial reasoning

Pro maintains coherent spatial relationships and object consistency across a scene —
the same character, furniture arrangement, or brand asset stays recognisable when the
prompt specifies spatial cues (`in front of`, `to the left of`, `reflected in`).

The prompt below combines three spatially related elements to test scene coherence.


In [ ]:
b64_scene = generate_image(
    prompt=(
        "Architect's desk with a laptop open to a blueprint, "
        "a scale model building sitting to the left of the laptop, "
        "a pencil resting across the blueprint in the foreground, "
        "natural overhead light, shallow depth of field, photorealistic"
    ),
    width=1024,
    height=1024,
)
show(b64_scene, save_as="03-scene.png")


## 7. Your Turn to Explore

Try pushing the model further on the capabilities explored above:

1. **Character consistency across edits** — generate a portrait, then use `edit_image`
   to change the background or lighting. Does the person remain recognisable?
2. **Longer text strings** — try rendering a 6–8 word phrase on a product label.
   Compare with a 2-word version to find the practical rendering limit.
3. **Layered spatial scenes** — add a fourth spatially referenced object to the
   architect's desk scene and observe whether the layout stays coherent.


In [ ]:
# Your experiments here


## 8. Summary

You used MAI-Image-2.5-Pro to:

- Generate a high-fidelity portrait using the 3:4 aspect ratio suited to head-and-shoulders framing
- Produce an image with readable embedded text (storefront signage)
- Build a multi-object scene relying on spatial relationships and object consistency

**Next steps**

- [MAI-Image-2.5 capsule](../mai-image-2.5/) — baseline generation and editing
- [MAI-Image-2.5-Flash capsule](../mai-image-2.5-flash/) — throughput-optimised variant

**References**

- [Deploy and use MAI image models in Microsoft Foundry](https://learn.microsoft.com/en-us/azure/foundry/foundry-models/how-to/use-foundry-models-mai-image?tabs=python)
- [Introducing MAI-Image-2.5-Pro and MAI-Voice-2-Flash](https://microsoft.ai/news/introducing-mai-image-2-5-pro-and-mai-voice-2-flash/)
- [MAI-Image-2.5 model page](https://microsoft.ai/models/mai-image-2-5/)
- [Image generation primer](../../../docs/primers/image-generation.md)
